# 03 · Vietnamese OCR on keyframes

Produces `derived/ocr/ocr.parquet`.

Likely the **highest-precision branch** for this corpus. Vietnamese news carries
headlines, tickers and name captions, and matching that text is close to an exact
match on the event — far sharper than visual similarity.

Only frames whose text is long enough to be meaningful are kept; single stray
glyphs are noise that would dilute BM25.

In [ ]:
# --- Colab setup -------------------------------------------------------------
# Runtime > Change runtime type > T4 GPU before running.
!nvidia-smi -L
!git clone -q https://github.com/YOUR_ORG/new_aic2026.git /content/aic || (cd /content/aic && git pull -q)
%cd /content/aic
!pip install -q pandas pyarrow pillow tqdm
import sys; sys.path.insert(0, "/content/aic/src")

In [ ]:
# --- Mount Drive -------------------------------------------------------------
# Keyframes go in, artifacts come out. Keeping both on Drive means an interrupted
# runtime resumes instead of restarting from zero.
from google.colab import drive

drive.mount('/content/drive')

from pathlib import Path

DATA = Path('/content/drive/MyDrive/aic2026')
KEYFRAMES = DATA / 'raw/keyframes'
DERIVED   = DATA / 'derived'
DERIVED.mkdir(parents=True, exist_ok=True)
print('keyframes:', KEYFRAMES, KEYFRAMES.exists())

In [ ]:
!pip install -q paddlepaddle-gpu paddleocr

from paddleocr import PaddleOCR

ocr = PaddleOCR(use_angle_cls=True, lang='vi', show_log=False)

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

MIN_CHARS = 4          # discard stray glyphs
MIN_CONFIDENCE = 0.60

catalog = pd.read_parquet(DERIVED / 'catalog.parquet')
OUT = DERIVED / 'ocr'; OUT.mkdir(parents=True, exist_ok=True)

def read_frame(path):
    image = Image.open(DATA / path).convert('RGB')
    result = ocr.ocr(np.array(image), cls=True)
    if not result or not result[0]:
        return ''
    parts = [text for _, (text, score) in result[0]
             if score >= MIN_CONFIDENCE and len(text.strip()) >= MIN_CHARS]
    return ' '.join(parts).strip()

In [ ]:
rows = []
for video_id, group in tqdm(catalog.groupby('video_id')):
    shard = OUT / f'{video_id}.parquet'
    if shard.exists():
        rows.append(pd.read_parquet(shard)); continue

    records = []
    for item in group.itertuples():
        text = read_frame(item.path)
        if not text:
            continue
        records.append({
            'video_id':    video_id,
            'text':        text,
            # One keyframe's text is attributed to that exact frame; the search
            # layer widens it to the surrounding shot when joining.
            'start_frame': int(item.frame_idx),
            'end_frame':   int(item.frame_idx),
            'start_time':  float(item.pts_time) if pd.notna(item.pts_time) else None,
            'end_time':    float(item.pts_time) if pd.notna(item.pts_time) else None,
        })

    table = pd.DataFrame(records)
    table.to_parquet(shard, index=False)
    rows.append(table)

ocr_table = pd.concat(rows, ignore_index=True)
ocr_table.to_parquet(OUT / 'ocr.parquet', index=False)
print(f'{len(ocr_table):,} frames with text')
ocr_table.head()

## Download

Copy to `data/derived/ocr/ocr.parquet`, then `aic build-text`.